# 07. Exportar CSV da aplicação Splink

Gera dois CSV a partir de `censo_limpo_aplicacao` e `cpf_limpo_aplicacao`
(mesmos parquets do predict do 02d), com as colunas na ordem pedida,
todas como string.

`nome_mae_phon` sai **como está no limpo** (é o que o Splink usou).
Sem coluna de mãe no bronze (`*_NOME_COL_MAE_PHON` vazios), o limpo calcula
a fonética da mãe com `phonetic_name_sql` (mesmo patch em
`gerar_registros_fonetica.py`). Rode o patch antes de exportar se o limpo
ainda tiver mãe = limpo sem fonética.

Saída em `OUTPUT_DIR`:
- `censo_aplicacao_splink.csv`
- `cpf_aplicacao_splink.csv`


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    CENSO_LIMPO_APLICACAO,
    CPF_LIMPO_APLICACAO,
    OUTPUT_DIR,
    ensure_output_dir,
    get_connection,
    print_paths,
    require_input,
)

print_paths()
ensure_output_dir()
require_input(CENSO_LIMPO_APLICACAO, label='CENSO_LIMPO_APLICACAO')
require_input(CPF_LIMPO_APLICACAO, label='CPF_LIMPO_APLICACAO')

con = get_connection()

COLS = [
    'unique_id',
    'nome_completo_phon',
    'primeiro_nome_phon',
    'ultimo_nome_phon',
    'data_nascimento',
    'ano_nascimento',
    'mes_nascimento',
    'dia_nascimento',
    'idade',
    'nome_mae_phon',
    'logradouro_norm',
    'cep',
    'cod_municipio',
    'sexo',
]

# Partes da data iguais ao materialize_splink_input.
# nome_mae_phon do limpo (já fonético após materialize/patch).
select_cols = '''
    CAST(unique_id AS VARCHAR) AS unique_id,
    CAST(nome_completo_phon AS VARCHAR) AS nome_completo_phon,
    CAST(primeiro_nome_phon AS VARCHAR) AS primeiro_nome_phon,
    CAST(ultimo_nome_phon AS VARCHAR) AS ultimo_nome_phon,
    CAST(data_nascimento AS VARCHAR) AS data_nascimento,
    CAST(
        CASE
            WHEN data_nascimento IS NULL
              OR TRIM(CAST(data_nascimento AS VARCHAR)) = '' THEN NULL
            ELSE substr(CAST(data_nascimento AS VARCHAR), 1, 4)
        END AS VARCHAR
    ) AS ano_nascimento,
    CAST(
        CASE
            WHEN data_nascimento IS NULL
              OR TRIM(CAST(data_nascimento AS VARCHAR)) = '' THEN NULL
            ELSE substr(CAST(data_nascimento AS VARCHAR), 6, 2)
        END AS VARCHAR
    ) AS mes_nascimento,
    CAST(
        CASE
            WHEN data_nascimento IS NULL
              OR TRIM(CAST(data_nascimento AS VARCHAR)) = '' THEN NULL
            ELSE substr(CAST(data_nascimento AS VARCHAR), 9, 2)
        END AS VARCHAR
    ) AS dia_nascimento,
    CAST(idade AS VARCHAR) AS idade,
    CAST(nome_mae_phon AS VARCHAR) AS nome_mae_phon,
    CAST(logradouro_norm AS VARCHAR) AS logradouro_norm,
    CAST(cep AS VARCHAR) AS cep,
    CAST(cod_municipio AS VARCHAR) AS cod_municipio,
    CAST(sexo AS VARCHAR) AS sexo
'''

out_censo = OUTPUT_DIR / 'censo_aplicacao_splink.csv'
out_cpf = OUTPUT_DIR / 'cpf_aplicacao_splink.csv'

censo_path = str(CENSO_LIMPO_APLICACAO).replace("'", "''")
cpf_path = str(CPF_LIMPO_APLICACAO).replace("'", "''")

con.execute(f'''
COPY (
    SELECT {select_cols}
    FROM read_parquet('{censo_path}')
) TO '{out_censo}' (HEADER, DELIMITER ',')
''')

con.execute(f'''
COPY (
    SELECT {select_cols}
    FROM read_parquet('{cpf_path}')
) TO '{out_cpf}' (HEADER, DELIMITER ',')
''')

n_censo = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{censo_path}')"
).fetchone()[0]
n_cpf = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{cpf_path}')"
).fetchone()[0]

print('Colunas:', COLS)
print('Censo CSV:', out_censo, '| linhas:', f'{n_censo:,}')
print('CPF CSV:  ', out_cpf, '| linhas:', f'{n_cpf:,}')

print('Amostra mãe no limpo (nome_mae vs nome_mae_phon)')
display(con.execute(f'''
SELECT nome_mae, nome_mae_phon
FROM read_parquet('{censo_path}')
WHERE nome_mae IS NOT NULL
  AND nome_mae_phon IS DISTINCT FROM nome_mae
LIMIT 5
''').df())

display(con.execute(f'''
SELECT * FROM read_csv('{out_censo}', header=true, auto_detect=true) LIMIT 3
''').df())
display(con.execute(f'''
SELECT * FROM read_csv('{out_cpf}', header=true, auto_detect=true) LIMIT 3
''').df())

con.close()
